#### Define notebook Parameters

Pass through th pl_worker

In [12]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
task_executions_id = ""
parent_run_id = ""
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
prev_wm = 'TODAY() - 30' or 1900-01-01
curr_wm = 'TODAY()'

# Framework-style placeholder; real connection settings should come from pipeline/metadata
source_connection_settings = "{}"

# Source config: one database, many tables to load
source_settings = json.dumps({
    "source_name": "CAN_RMDD",
    "database_name": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
    "table_config": [
        {
            "schema_name": "dbo",
            "table_name": "Address",
            "target_table": "silver_rmdd_address",
            "primary_keys": ["AddressID"],
            # "select_columns": [
            #     "AddressID",
            #     "AddressTypeCode",
            #     "Address1",
            #     "Address2",
            #     "Address3",
            #     "Address4",
            #     "Address5",
            #     "City",
            #     "CountryID",
            #     "State",
            #     "PostalCode",
            #     "Active",
            #     "LastUpdatedDateGMT"
            # ],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "Client",
            "target_table": "silver_rmdd_client",
            "primary_keys": ["ClientNumber", "MemberFirmCode"],
            # "select_columns": [
            #     "ClientID",
            #     "ClientNumber",
            #     "MemberFirmCode",
            #     "PMSClientID",
            #     "ClientName",
            #     "AlternateClientName",
            #     "ContactName",
            #     "OpenDateGMT",
            #     "CloseDateGMT",
            #     "ClientDUNS",
            #     "ClientGUDUNS",
            #     "Comments",
            #     "PMSEntityType",
            #     "ClientType",
            #     "IsConfidential",
            #     "SICCode",
            #     "SanctionsChecked_YN",
            #     "Active",
            #     "CreatedDateGMT",
            #     "LastUpdatedDateGMT"
            # ],
            "is_incremental": 0,
            "incremental_column": "LastUpdatedDateGMT"
        }
    ]
})

# Target config: common silver destination settings
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_rmdd",
    "load_strategy": "merge",
    "location_root": "Files/silver/rmdd"
})

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 18, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [13]:
%run nb_transactional_shared_functions

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 23, Finished, Available, Finished, True)

In [14]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 24, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [15]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("database_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge")
location_root = target_settings.get("location_root")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
)

# Temportal to replace
if not server_name:
    server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"

print(source_name)
print(source_database)
print(target_schema)
display(table_config)

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 25, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd


SynapseWidget(Synapse.DataFrame, 8ac1de16-7045-463f-9075-2de4103f7b57)

#### Build JDBC connection

Connects to source SQL database

In [16]:
# Build one JDBC connection for the source SQL database
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(jdbc_url)
print(connection_properties)

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 26, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;
{'accessToken': '[REDACTED]', 'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'}


#### Ingest source table

Read and dedup clean source

In [17]:
# Read each source table and drop exact duplicates only
for cfg in table_config:
    # Get variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    column_list = cfg.get("select_columns", ["*"])
    is_incremental = cfg.get("is_incremental", 0)
    watermark_col = cfg.get("incremental_column")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    column_list_sql = ", ".join(column_list)
    source_view = f"src_{source_table.lower()}"

    # Set watermark based on YAML watermark_col
    watermark_filter = "1 = 1"
    if is_incremental == 1 and prev_wm and curr_wm and watermark_col:
        watermark_filter = f"{watermark_col} > '{prev_wm}' AND {watermark_col} <= '{curr_wm}'"

    # Query to source with watermark applied
    source_query = f"""
    (
        SELECT {column_list_sql}
        FROM {full_source_name}
        WHERE {watermark_filter}
    ) AS src
    """

    # Read table using JDBC connection
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    # Remove duplicates and create temp view for later consumption
    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)

    print(f"Source preview for {full_source_name}")
    display(df_source.limit(3))

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 27, Finished, Available, Finished, False)

Source preview for dbo.Address


SynapseWidget(Synapse.DataFrame, e9c2d111-99de-455f-adb2-e939757a8f8a)

Source preview for dbo.Client


SynapseWidget(Synapse.DataFrame, 077695c5-1a17-4582-a39b-73634ac14917)

#### Transform source table

Applies mapping and transformation as needed

In [18]:
# Map source columns into the target shape
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    source_view = f"src_{source_table.lower()}"
    target_view = f"tgt_{source_table.lower()}"

    # Transformation block
    if source_table == "Address":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY AddressID) AS BIGINT)  AS address_key,
            AddressID               AS address_id,
            AddressTypeCode         AS address_type_code,
            Address1                AS address_1,
            Address2                AS address_2,
            Address3                AS address_3,
            Address4                AS address_4,
            Address5                AS address_5,
            City                    AS city,
            CountryID               AS country_id,
            CAST(NULL AS STRING)    AS country_code_iso2,
            State                   AS state,
            PostalCode              AS zip_code,
            Address1                AS street_address_check,
            Active                  AS is_active,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "Client":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY ClientID) AS BIGINT)   AS client_key,
            ClientNumber            AS client_number,
            MemberFirmCode          AS member_firm_code,
            ClientID                AS client_id,
            PMSClientID             AS pms_client_id,
            ClientName              AS client_name,
            AlternateClientName     AS alternate_client_name,
            ContactName             AS contact_name,
            OpenDateGMT             AS open_date_utc,
            CloseDateGMT            AS close_date_utc,
            ClientDUNS              AS client_duns,
            ClientGUDUNS            AS client_guduns,
            Comments                AS comments,
            PMSEntityType           AS pms_entity_type,
            ClientType              AS client_type,
            IsConfidential          AS is_confidential,
            SICCode                 AS sic_code,
            CAST(NULL AS STRING)    AS client_sector_key,
            SanctionsChecked_YN     AS is_sanctions_checked,
            Active                  AS is_active,
            CreatedDateGMT          AS created_date_utc_at_source,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    else:
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT *
        FROM {source_view}
        """)

    print(f"Mapped preview for {target_view}")
    display(spark.table(target_view).limit(3))

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 28, Finished, Available, Finished, False)

Mapped preview for tgt_address


SynapseWidget(Synapse.DataFrame, d38b3c77-0133-4cab-a6b5-e16989234e7c)

Mapped preview for tgt_client


SynapseWidget(Synapse.DataFrame, f4323bb9-070f-4a4c-89ab-b9ee2341b8e0)

#### Add metadata to source table

Applies metadata and _hashed_pk

In [19]:
# Add metadata and hashed key to the transformed tgt_* views
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    target_view = f"tgt_{source_table.lower()}"

    # Add metadata to source table
    df_target = spark.table(target_view)
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=full_source_name,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Get primary columns based on YAML file
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )
    df_target.createOrReplaceTempView(target_view)

    print(f"Final preview for {full_source_name}")
    display(df_target.limit(3))

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 29, Finished, Available, Finished, False)

Final preview for dbo.Address


SynapseWidget(Synapse.DataFrame, 0ef51885-1d8c-4176-ae3d-b05803223e8f)

Final preview for dbo.Client


SynapseWidget(Synapse.DataFrame, fcd89faf-7117-4e9b-aa9c-284c91fddf1b)

#### Check duplicates

Return duplicates before merge 

In [20]:
# Check for duplicate hashed keys before merge
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_view = f"tgt_{source_table.lower()}"

    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 30, Finished, Available, Finished, False)

Checking duplicates for tgt_address
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 94ca02a9-486b-48f9-bbfc-edebfbd3a027)

Checking duplicates for tgt_client
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, ccbabddc-e604-4de7-be27-eab531db3106)

#### Merge to target

Merge to target table based on _hashed_key

In [21]:
# Merge each final temp view into its target silver table
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    target_view = f"tgt_{source_table.lower()}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Write delta files to path
    df_target = spark.table(target_view)
    load_to_target(df_target, target_table_path, target_load_strategy)

    # Create table based on delta files
    # spark.sql(f"""
    # CREATE TABLE IF NOT EXISTS {full_target_name}
    # USING DELTA
    # LOCATION '{target_table_path}'
    # """)

    # Return operationMetrics > # rows affected
    delta_table = DeltaTable.forPath(spark, target_table_path)
    operation_metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]

    rows_inserted = int(operation_metrics.get("numTargetRowsInserted", operation_metrics.get("numOutputRows", 0)))
    rows_updated = int(operation_metrics.get("numTargetRowsUpdated", 0))
    rows_affected = rows_inserted + rows_updated

    print(full_target_name)
    print(f"rows_affected: {rows_affected}")

StatementMeta(, cab14fd7-c27f-456b-bf2b-2fcc8cec552b, 31, Finished, Available, Finished, False)

2026-06-30 16:43:39,276 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_address (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_address
rows_affected: 252665
2026-06-30 16:43:51,739 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_client (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_client
rows_affected: 196838
